# Distance Metric Learning: aligning z_g and theta_scaled geometry

Learn two Mahalanobis matrices, `A` for `z_g` (graph embeddings) and `B` for `theta_scaled`
(scaled ABCD parameters), such that pairwise distances under `A` in embedding space match
pairwise distances under `B` in parameter space as closely as possible for the *same*
instances. Intuition: this finds a (possibly rank-reduced, anisotropic) metric on each space
under which "close graphs" <-> "close parameters".

- `A = Lz @ Lz.T`, `B = Lt @ Lt.T` (free square matrices, so PSD by construction).
- Trained via mini-batches of matched instance indices (full 8740x8740 pairwise distance
  matrices aren't needed every step).
- Distances are mean-normalised per batch before comparison, since otherwise `A = B = 0`
  trivially minimises the raw distance gap — this makes the loss scale-invariant and only
  sensitive to the *shape* of the two geometries.

## Required `embeddings.pt` structure

This notebook expects `EMBEDDINGS_DATA_PATH` to point at a `torch.save`d `dict` with (at least)
the following row-aligned keys, for `N` instances:

| key            | type                | shape     | description                                          |
| -------------- | ------------------- | --------- | ----------------------------------------------------- |
| `z_g`          | `torch.FloatTensor` | `(N, 64)` | graph encoder embeddings                               |
| `theta_scaled` | `torch.FloatTensor` | `(N, 9)`  | scaled ABCD parameters, columns ordered per `THETA_NAMES` |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

In [ ]:
EMBEDDINGS_DATA_PATH = Path("../../DCBA-data-set/data/embeddings/embeddings.pt")
THETA_NAMES = ["n", "t1", "t2", "xi", "c_min", "c_max", "d_min", "d_max", "nout"]

In [ ]:
data = torch.load(EMBEDDINGS_DATA_PATH, map_location="cpu", weights_only=False)

In [ ]:
torch.manual_seed(0)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device = {device}")

Z = data["z_g"].to(device=device, dtype=torch.float32)
T = data["theta_scaled"].to(device=device, dtype=torch.float32)

N, dz = Z.shape
_, dt = T.shape

# row-wise train/val split (val used only to check generalisation of the learned metrics)
perm = torch.randperm(N)
n_val = int(0.1 * N)
val_idx = perm[:n_val]
train_idx = perm[n_val:]

print(f"N={N}  dz={dz}  dt={dt}  train={len(train_idx)}  val={len(val_idx)}")


class MahalanobisPair(nn.Module):
    """
    Learns paired Mahalanobis metrics ``A`` (over ``z_g``) and ``B`` (over ``theta_scaled``).

    Each metric is parametrised by a free square matrix ``L`` such that ``A = Lz @ Lz.T`` and
    ``B = Lt @ Lt.T``, which makes both positive semi-definite by construction.

    :param dz: Dimensionality of the ``z_g`` embedding space.
    :param dt: Dimensionality of the ``theta_scaled`` parameter space.
    """

    def __init__(self, dz: int, dt: int) -> None:
        """Initialise both metric factors as identity matrices."""
        super().__init__()
        self.Lz = nn.Parameter(torch.eye(dz))
        self.Lt = nn.Parameter(torch.eye(dt))

    @staticmethod
    def _pairwise_sqdist(X: torch.Tensor, L: torch.Tensor) -> torch.Tensor:
        """
        Compute pairwise squared Mahalanobis distances under metric ``L @ L.T``.

        :param X: Row-wise data matrix of shape ``(batch, dim)``.
        :param L: Metric factor of shape ``(dim, dim)``.

        :returns: Pairwise squared distance matrix of shape ``(batch, batch)``.
        """
        Y = X @ L  # A = L @ L.T  =>  ||x_i - x_j||_A^2 == ||y_i - y_j||_2^2
        return torch.cdist(Y, Y, p=2).pow(2)

    def forward(self, Xz: torch.Tensor, Xt: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Compute pairwise squared distances in both spaces for the same batch of instances.

        :param Xz: Batch of ``z_g`` embeddings, shape ``(batch, dz)``.
        :param Xt: Batch of ``theta_scaled`` values, shape ``(batch, dt)``.

        :returns: Tuple of ``(Dz, Dt)`` pairwise squared distance matrices.
        """
        Dz = self._pairwise_sqdist(Xz, self.Lz)
        Dt = self._pairwise_sqdist(Xt, self.Lt)
        return Dz, Dt

    @property
    def A(self) -> torch.Tensor:
        """Mahalanobis metric over the ``z_g`` embedding space."""
        return self.Lz @ self.Lz.T

    @property
    def B(self) -> torch.Tensor:
        """Mahalanobis metric over the ``theta_scaled`` parameter space."""
        return self.Lt @ self.Lt.T


def scale_invariant_loss(Dz: torch.Tensor, Dt: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Mean-normalise both distance matrices before comparing, avoiding the trivial ``A = B = 0``.

    :param Dz: Pairwise squared distances under metric ``A``.
    :param Dt: Pairwise squared distances under metric ``B``.
    :param eps: Small constant added to the mean to avoid division by zero.

    :returns: Scalar loss tensor.
    """
    Dz_n = Dz / (Dz.mean() + eps)
    Dt_n = Dt / (Dt.mean() + eps)
    return (Dz_n - Dt_n).pow(2).mean()


model = MahalanobisPair(dz, dt).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

**Training**

In [ ]:
n_steps = 2000
batch_size = 256
log_every = 200

history = []
for step in range(1, n_steps + 1):
    b = train_idx[torch.randint(0, len(train_idx), (batch_size,))]
    Xz, Xt = Z[b], T[b]

    Dz, Dt = model(Xz, Xt)
    loss = scale_invariant_loss(Dz, Dt)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % log_every == 0 or step == 1:
        history.append((step, loss.item()))
        print(f"step {step:5d}  loss {loss.item():.5f}")

**Validation**

In [ ]:
def pearson_corr(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Compute the Pearson correlation coefficient between two flattened tensors.

    :param a: First tensor.
    :param b: Second tensor.

    :returns: Scalar correlation coefficient.
    """
    a = a - a.mean()
    b = b - b.mean()
    return (a * b).sum() / (a.norm() * b.norm() + 1e-8)


model.eval()
with torch.no_grad():
    Xz_val, Xt_val = Z[val_idx], T[val_idx]
    Dz_val, Dt_val = model(Xz_val, Xt_val)
    val_loss = scale_invariant_loss(Dz_val, Dt_val)

    # off-diagonal entries only (diagonal is identically 0 and would inflate the correlation)
    n_v = Dz_val.shape[0]
    off_diag = ~torch.eye(n_v, dtype=torch.bool, device=device)
    dz_flat = Dz_val[off_diag]
    dt_flat = Dt_val[off_diag]

    corr = pearson_corr(dz_flat, dt_flat)

print(f"val loss (scale-invariant MSE): {val_loss.item():.5f}")
print(f"val Pearson corr(d_A(z_g), d_B(theta_scaled)): {corr.item():.4f}")
_ = model.train()

### Interpretation

`B` (9x9, over `theta_scaled`) is small enough to read directly: its eigenvectors show which
combinations of ABCD input parameters the embedding geometry is most sensitive to, and the
eigenvalues show how strongly. Rows/columns of `B` follow the `THETA_NAMES` order. `A` (64x64,
over `z_g`) is summarised by its eigenvalue spectrum (effective rank) instead.

In [ ]:
with torch.no_grad():
    A = model.A.detach().cpu()
    B = model.B.detach().cpu()

    evals_A = torch.linalg.eigvalsh(A).flip(0)
    evals_B, evecs_B = torch.linalg.eigh(B)
    evals_B = evals_B.flip(0)
    evecs_B = evecs_B.flip(1)

print("A (z_g metric) eigenvalues, top 10 / 64:")
print([round(v, 4) for v in evals_A[:10].tolist()])

print("\nB (theta_scaled metric) eigenvalues (all 9):")
print([round(v, 4) for v in evals_B.tolist()])

print("\nB eigenvectors (columns), each row = one theta_scaled dim:")
for name, row in zip(THETA_NAMES, evecs_B.tolist()):
    print(f"{name:>6s}", ["{:+.3f}".format(v) for v in row])

**Vis**

In [ ]:
%matplotlib inline

steps, losses = zip(*history)
plt.figure(figsize=(6, 4))
plt.plot(steps, losses, marker="o", ms=3)
plt.xlabel("training step")
plt.ylabel("scale-invariant MSE loss")
plt.title("Training loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# subsample off-diagonal validation pairs for a legible scatter (dz_flat/dt_flat has ~760k points)
n_plot = min(8000, dz_flat.numel())
plot_sel = torch.randperm(dz_flat.numel())[:n_plot]
x = dz_flat[plot_sel].cpu().numpy()
y = dt_flat[plot_sel].cpu().numpy()

plt.figure(figsize=(5, 5))
plt.scatter(x, y, s=4, alpha=0.25, linewidths=0)
plt.xlabel("d_A(z_g, z_g')  [Mahalanobis, embedding space]")
plt.ylabel("d_B(theta, theta')  [Mahalanobis, parameter space]")
plt.title(f"Held-out pairwise distances (corr = {corr.item():.3f})")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].bar(range(1, 21), evals_A[:20].numpy())
axes[0].set_xlabel("eigenvalue rank")
axes[0].set_ylabel("eigenvalue")
axes[0].set_title("A (z_g metric) spectrum, top 20/64")
axes[0].grid(alpha=0.3)

axes[1].bar(range(1, dt + 1), evals_B.numpy())
axes[1].set_xlabel("eigenvalue rank")
axes[1].set_title("B (theta_scaled metric) spectrum, all 9")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(5.5, 5))
im = plt.imshow(evecs_B.numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, label="loading")
plt.xlabel("eigenvector (0 = largest eigenvalue)")
plt.ylabel("theta_scaled dim")
plt.xticks(range(dt), [str(i) for i in range(dt)])
plt.yticks(range(dt), THETA_NAMES)
plt.title("B eigenvector loadings")
plt.tight_layout()
plt.show()

### Reading the `B` loadings

The spectrum is highly concentrated: the top eigenvalue (10.20) alone carries ~49% of the
trace, and the top 3 carry ~82%, leaving the bottom 5 components with only ~11.5% combined.
So although `theta_scaled` is 9-dimensional, `B` behaves as if it were closer to rank 3-4 —
the learned metric is telling us most `theta_scaled` dimensions barely matter for matching
`z_g` distances.

- **PC0 (λ=10.20, ~49%)** is almost pure `t1` (loading -0.997). The degree power-law exponent
  gets its own dedicated, essentially unmixed axis — by far the single most important
  direction.
- **PC2 (λ=2.62, ~13%)** is almost pure `t2` (+0.996), the community-size power-law exponent.
  Like `t1`, it stands alone rather than blending with other parameters.
- **PC1 (λ=4.29, ~21%)** and **PC3 (λ=1.27, ~6%)** are both dominated by `n` and `d_min`
  mixed together (PC1: `n` +0.811 / `d_min` -0.571; PC3: `n` -0.531 / `d_min` -0.775). Unlike
  `t1`/`t2`, graph size and minimum degree aren't separated into their own axes — the metric
  treats them as entangled, spreading their joint contribution across two components.
- **PC4-PC8 (λ from 0.65 down to 0.30, ~11.5% combined)** mix `xi`, `c_min`, `c_max`, `d_max`,
  `nout` together with no dominant single-parameter direction in any of them. These five
  community-size/noise-related parameters barely register individually — from the geometry
  `B` learned, they're largely redundant with each other and contribute comparatively little
  to matching `z_g`.

This is qualitatively consistent with `A`'s spectrum above (two dominant eigenvalues, 158.78
and 115.47, then a steep drop to a long tail): both learned metrics independently concentrate
on a low-dimensional subspace, which is reassuring given the val correlation of 0.933. It's
worth treating the smallest `B` eigenvalues (and their eigenvectors) with some caution,
though — at this scale they may reflect noise near convergence rather than a robust signal,
so the `xi`/`c_min`/`c_max`/`d_max`/`nout` groupings above are less trustworthy than the
`t1`/`t2`/`(n, d_min)` structure in the top 4 components.
